In [ ]:
# ==================================================================================================
# FINAL H=10 ZERO-SHOT CROSS-EXPERIMENT PERFORMANCE EVALUATION
#
# Independent / previously unseen experiments:
#   Exp. 12
#   Exp. 13
#   Exp. 14
#
# Inputs:
#   1. Final H=10 Fusion-LSTM prediction CSVs
#   2. Independent COCO ground-truth annotations
#
# Evaluation:
#   Prediction generated at frame t
#                 ↓
#   H = 10 future target
#                 ↓
#   Compare prediction against independently annotated frame t+10
#
# IMPORTANT:
#   - No training is performed.
#   - No scaler is refitted.
#   - No threshold is optimized.
#   - No prediction is altered.
#   - Only explicitly annotated COCO frames are evaluated.
#   - Unannotated frames are NEVER assumed to be Good.
# ==================================================================================================

from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

ROOT = Path.cwd()

PRED_DIR = (
    ROOT
    / "external_test"
    / "external_test_outputs_H10_full_features_FINAL"
)

PRED_FILES = {
    "exp_12": PRED_DIR / "exp_12_H10_predictions_full_features.csv",
    "exp_13": PRED_DIR / "exp_13_H10_predictions_full_features.csv",
    "exp_14": PRED_DIR / "exp_14_H10_predictions_full_features.csv",
}

# Final independent COCO ground truth
COCO_JSON = Path(
    "data/external/_annotations.coco.json"
)

OUTPUT_DIR = (
    ROOT
    / "external_test"
    / "FINAL_H10_ZERO_SHOT_EXP12_EXP13_EXP14"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

EXPECTED_HORIZON = 10

CLASS_ORDER = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

BINARY_ORDER = [
    "Good",
    "Defect",
]


# ==================================================================================================
# 3. COCO CATEGORY MAPPING
# ==================================================================================================

# COCO segmentation names -> manuscript / Fusion-LSTM target names

CATEGORY_MAPPING = {

    # Burr
    "burr": "Burr",
    "burrs": "Burr",

    # Flash-burr
    "flash_bur": "Flash-burr",
    "flash_burr": "Flash-burr",
    "flash-burr": "Flash-burr",

    # Surface groove / void
    "surface_groove_void": "Surface-groove/void",
    "surface-groove-void": "Surface-groove/void",
    "surface_groove/void": "Surface-groove/void",
    "surface-groove/void": "Surface-groove/void",
}

# Categories that do NOT themselves define a defect state
NON_DEFECT_CATEGORIES = {
    "weld",
    "fusion-model-validation",
}


# ==================================================================================================
# 4. HELPER FUNCTIONS
# ==================================================================================================

def normalize_text(value):
    return str(value).strip().lower()


def normalize_prediction_class(value):
    """
    Normalize Fusion-LSTM output labels into the four
    manuscript classes.
    """

    value = normalize_text(value)

    mapping = {
        "good": "Good",

        "burr": "Burr",
        "burrs": "Burr",

        "flash-burr": "Flash-burr",
        "flash_burr": "Flash-burr",
        "flash-bur": "Flash-burr",
        "flash_bur": "Flash-burr",

        "surface-groove/void": "Surface-groove/void",
        "surface_groove_void": "Surface-groove/void",
        "surface-groove-void": "Surface-groove/void",
    }

    return mapping.get(
        value,
        str(value),
    )


def extract_experiment_and_frame(filename):
    """
    Robustly extract experiment and original frame number
    from Roboflow-style filenames.

    Examples:

    Exp_12_000207_jpg.rf.cb1e495e....jpg
        -> exp_12, 207

    EXP_14_000375_jpg.rf.730598....jpg
        -> exp_14, 375

    Exp_13_000298.jpg
        -> exp_13, 298
    """

    text = Path(filename).name

    pattern = re.compile(
        r"exp[_\-]?(\d+)[_\-](\d+)",
        flags=re.IGNORECASE,
    )

    match = pattern.search(text)

    if match is None:
        return None, None

    exp_number = int(
        match.group(1)
    )

    frame_number = int(
        match.group(2)
    )

    return (
        f"exp_{exp_number}",
        frame_number,
    )


def calculate_metrics(
    y_true,
    y_pred,
):
    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_precision,
        "weighted_recall": weighted_recall,
        "weighted_f1": weighted_f1,
    }


# ==================================================================================================
# 5. VERIFY INPUT FILES
# ==================================================================================================

print("=" * 120)
print("FINAL H=10 ZERO-SHOT CROSS-EXPERIMENT PERFORMANCE EVALUATION")
print("=" * 120)

print("\nPrediction files:")

for exp_name, path in PRED_FILES.items():

    print(f"\n{exp_name}:")
    print(path)

    if not path.exists():
        raise FileNotFoundError(
            f"\nPrediction file not found:\n{path}"
        )


print("\nCOCO ground truth:")
print(COCO_JSON)

if not COCO_JSON.exists():
    raise FileNotFoundError(
        f"\nCOCO JSON file not found:\n{COCO_JSON}"
    )


print("\nAll required input files found.")


# ==================================================================================================
# 6. LOAD COCO JSON
# ==================================================================================================

print("\n" + "=" * 120)
print("LOADING INDEPENDENT COCO GROUND TRUTH")
print("=" * 120)

with open(
    COCO_JSON,
    "r",
    encoding="utf-8",
) as file:

    coco = json.load(file)


images = coco.get(
    "images",
    [],
)

annotations = coco.get(
    "annotations",
    [],
)

categories = coco.get(
    "categories",
    [],
)


print(f"\nImages      : {len(images)}")
print(f"Annotations : {len(annotations)}")
print(f"Categories  : {len(categories)}")


# ==================================================================================================
# 7. COCO CATEGORY AUDIT
# ==================================================================================================

category_id_to_name = {
    category["id"]: category["name"]
    for category in categories
}


print("\nCOCO category mapping:")

for category_id, category_name in category_id_to_name.items():

    print(
        f"  {category_id}: {category_name}"
    )


# ==================================================================================================
# 8. COUNT ACTUAL ANNOTATION CATEGORY USAGE
# ==================================================================================================

annotation_category_counts = {}

for ann in annotations:

    category_name = category_id_to_name.get(
        ann["category_id"],
        "UNKNOWN",
    )

    annotation_category_counts[
        category_name
    ] = (
        annotation_category_counts.get(
            category_name,
            0,
        )
        + 1
    )


print("\nAnnotation counts by COCO category:")

for category_name, count in annotation_category_counts.items():

    print(
        f"  {category_name}: {count}"
    )


# ==================================================================================================
# 9. GROUP ANNOTATIONS BY IMAGE
# ==================================================================================================

annotations_by_image = {}

for ann in annotations:

    image_id = ann["image_id"]

    annotations_by_image.setdefault(
        image_id,
        [],
    ).append(
        ann
    )


# ==================================================================================================
# 10. BUILD FRAME-LEVEL COCO GROUND TRUTH
# ==================================================================================================

gt_rows = []

multi_defect_frames = []

unknown_category_frames = []

filename_parse_failures = []


for img in images:

    image_id = img["id"]

    filename = img["file_name"]

    (
        exp_name,
        frame_idx,
    ) = extract_experiment_and_frame(
        filename
    )


    # ------------------------------------------------------------------
    # Filename audit
    # ------------------------------------------------------------------

    if exp_name is None or frame_idx is None:

        filename_parse_failures.append(
            filename
        )

        continue


    # Only Exp. 12, 13, 14
    if exp_name not in PRED_FILES:
        continue


    image_annotations = annotations_by_image.get(
        image_id,
        [],
    )


    mapped_defect_classes = []

    original_category_names = []

    unknown_on_this_frame = []


    for ann in image_annotations:

        category_name = category_id_to_name.get(
            ann["category_id"],
            "UNKNOWN",
        )

        original_category_names.append(
            category_name
        )

        normalized_category = normalize_text(
            category_name
        )


        # --------------------------------------------------------------
        # Structural / non-defect category
        # --------------------------------------------------------------

        if normalized_category in NON_DEFECT_CATEGORIES:
            continue


        # --------------------------------------------------------------
        # Defect category
        # --------------------------------------------------------------

        mapped_class = CATEGORY_MAPPING.get(
            normalized_category
        )

        if mapped_class is not None:

            mapped_defect_classes.append(
                mapped_class
            )

        else:

            unknown_on_this_frame.append(
                category_name
            )


    mapped_defect_classes = sorted(
        set(
            mapped_defect_classes
        )
    )

    unknown_on_this_frame = sorted(
        set(
            unknown_on_this_frame
        )
    )


    # ------------------------------------------------------------------
    # Unknown categories must NOT silently become Good
    # ------------------------------------------------------------------

    if len(
        unknown_on_this_frame
    ) > 0:

        gt_class = None

        unknown_category_frames.append({
            "exp_name": exp_name,
            "frame_idx": frame_idx,
            "filename": filename,
            "unknown_categories":
                "|".join(
                    unknown_on_this_frame
                ),
            "all_categories":
                "|".join(
                    original_category_names
                ),
        })


    # ------------------------------------------------------------------
    # Explicitly no defect annotation -> Good
    # ------------------------------------------------------------------

    elif len(
        mapped_defect_classes
    ) == 0:

        gt_class = "Good"


    # ------------------------------------------------------------------
    # Exactly one defect category
    # ------------------------------------------------------------------

    elif len(
        mapped_defect_classes
    ) == 1:

        gt_class = (
            mapped_defect_classes[0]
        )


    # ------------------------------------------------------------------
    # More than one defect class
    # Cannot automatically assign one multiclass label
    # ------------------------------------------------------------------

    else:

        gt_class = None

        multi_defect_frames.append({
            "exp_name": exp_name,
            "frame_idx": frame_idx,
            "filename": filename,
            "defect_classes":
                "|".join(
                    mapped_defect_classes
                ),
            "all_categories":
                "|".join(
                    original_category_names
                ),
        })


    gt_rows.append({
        "exp_name": exp_name,
        "frame_idx": frame_idx,
        "image_id": image_id,
        "filename": filename,
        "gt_class": gt_class,
        "category_names":
            "|".join(
                original_category_names
            ),
        "number_of_defect_classes":
            len(
                mapped_defect_classes
            ),
        "has_unknown_category":
            len(
                unknown_on_this_frame
            ) > 0,
    })


gt_df = pd.DataFrame(
    gt_rows
)


# ==================================================================================================
# 11. COCO FRAME-LEVEL AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("COCO FRAME-LEVEL GROUND-TRUTH AUDIT")
print("=" * 120)


print(
    f"\nTotal Exp. 12-14 annotated images: "
    f"{len(gt_df)}"
)


print("\nAnnotated image counts by experiment:")

print(
    gt_df.groupby(
        "exp_name"
    ).size()
)


print("\nAnnotated frame ranges:")

for exp_name in sorted(
    gt_df["exp_name"].unique()
):

    temp = gt_df[
        gt_df["exp_name"]
        == exp_name
    ]

    print(
        f"{exp_name}: "
        f"{temp['frame_idx'].min()} "
        f"-> "
        f"{temp['frame_idx'].max()} "
        f"| n={len(temp)}"
    )


print("\nFrame-level ground-truth distribution:")

print(
    gt_df["gt_class"]
    .value_counts(
        dropna=False
    )
)


# ==================================================================================================
# 12. FILENAME PARSE AUDIT
# ==================================================================================================

print(
    "\nFilename parse failures:",
    len(
        filename_parse_failures
    )
)

if filename_parse_failures:

    failure_df = pd.DataFrame({
        "filename":
            filename_parse_failures
    })

    failure_path = (
        OUTPUT_DIR
        / "filename_parse_failures.csv"
    )

    failure_df.to_csv(
        failure_path,
        index=False,
    )

    print(
        f"Saved:\n{failure_path}"
    )


# ==================================================================================================
# 13. UNKNOWN CATEGORY AUDIT
# ==================================================================================================

print(
    "\nFrames containing unmapped COCO categories:",
    len(
        unknown_category_frames
    )
)

if unknown_category_frames:

    unknown_df = pd.DataFrame(
        unknown_category_frames
    )

    unknown_path = (
        OUTPUT_DIR
        / "unknown_category_frames_REVIEW.csv"
    )

    unknown_df.to_csv(
        unknown_path,
        index=False,
    )

    print(
        f"\nSaved for review:\n{unknown_path}"
    )


# ==================================================================================================
# 14. MULTIPLE DEFECT AUDIT
# ==================================================================================================

print(
    "\nFrames containing multiple mapped defect classes:",
    len(
        multi_defect_frames
    )
)

if multi_defect_frames:

    multi_df = pd.DataFrame(
        multi_defect_frames
    )

    multi_path = (
        OUTPUT_DIR
        / "multi_defect_frames_REVIEW.csv"
    )

    multi_df.to_csv(
        multi_path,
        index=False,
    )

    print(
        f"\nSaved for review:\n{multi_path}"
    )


# ==================================================================================================
# 15. DUPLICATE COCO EXPERIMENT / FRAME CHECK
# ==================================================================================================

duplicate_gt = (
    gt_df
    .duplicated(
        subset=[
            "exp_name",
            "frame_idx",
        ],
        keep=False,
    )
)


duplicate_gt_count = int(
    duplicate_gt.sum()
)


print(
    "\nDuplicate COCO (experiment, frame) records:",
    duplicate_gt_count
)


if duplicate_gt_count > 0:

    duplicate_gt_df = (
        gt_df[
            duplicate_gt
        ]
        .sort_values(
            [
                "exp_name",
                "frame_idx",
            ]
        )
    )

    duplicate_gt_path = (
        OUTPUT_DIR
        / "duplicate_COCO_frames_REVIEW.csv"
    )

    duplicate_gt_df.to_csv(
        duplicate_gt_path,
        index=False,
    )

    print(
        f"\nSaved for review:\n{duplicate_gt_path}"
    )


# ==================================================================================================
# 16. LOAD + AUDIT FINAL H=10 PREDICTIONS
# ==================================================================================================

print("\n" + "=" * 120)
print("FINAL H=10 PREDICTION AUDIT")
print("=" * 120)


prediction_frames = []


for expected_exp, path in PRED_FILES.items():

    df = pd.read_csv(
        path
    )

    required_columns = {
        "exp_name",
        "frame_idx",
        "target_frame",
        "predicted_class",
    }


    missing_columns = (
        required_columns
        - set(
            df.columns
        )
    )


    if missing_columns:

        raise ValueError(
            f"\nMissing required columns in:\n"
            f"{path}\n"
            f"{sorted(missing_columns)}"
        )


    df = df.copy()


    df["exp_name"] = (
        df["exp_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )


    df["frame_idx"] = pd.to_numeric(
        df["frame_idx"],
        errors="raise",
    ).astype(
        int
    )


    df["target_frame"] = pd.to_numeric(
        df["target_frame"],
        errors="raise",
    ).astype(
        int
    )


    # ------------------------------------------------------------------
    # Horizon audit
    # ------------------------------------------------------------------

    df["observed_horizon"] = (
        df["target_frame"]
        - df["frame_idx"]
    )


    observed_horizons = sorted(
        df[
            "observed_horizon"
        ]
        .unique()
        .tolist()
    )


    print(
        f"\n{expected_exp}"
    )

    print(
        f"  Prediction rows : {len(df)}"
    )

    print(
        f"  Target offsets  : {observed_horizons}"
    )


    if observed_horizons != [
        EXPECTED_HORIZON
    ]:

        raise RuntimeError(
            f"\n{expected_exp}: H=10 audit FAILED."
        )


    # ------------------------------------------------------------------
    # Experiment identity audit
    # ------------------------------------------------------------------

    observed_experiments = sorted(
        df[
            "exp_name"
        ]
        .unique()
        .tolist()
    )


    if observed_experiments != [
        expected_exp
    ]:

        raise RuntimeError(
            f"\nExperiment identity mismatch in "
            f"{path.name}\n"
            f"Expected: {expected_exp}\n"
            f"Observed: {observed_experiments}"
        )


    # ------------------------------------------------------------------
    # Prediction class normalization
    # ------------------------------------------------------------------

    df[
        "predicted_class_original"
    ] = df[
        "predicted_class"
    ]


    df[
        "predicted_class"
    ] = df[
        "predicted_class"
    ].map(
        normalize_prediction_class
    )


    invalid_prediction_classes = sorted(
        set(
            df[
                "predicted_class"
            ]
        )
        - set(
            CLASS_ORDER
        )
    )


    if invalid_prediction_classes:

        raise RuntimeError(
            "\nUnknown predicted classes found:\n"
            f"{invalid_prediction_classes}"
        )


    print(
        "  H=10 audit     : PASSED"
    )


    prediction_frames.append(
        df
    )


pred_df = pd.concat(
    prediction_frames,
    ignore_index=True,
)


print(
    f"\nTotal final H=10 prediction rows: "
    f"{len(pred_df)}"
)


# ==================================================================================================
# 17. DUPLICATE PREDICTION TARGET CHECK
# ==================================================================================================

duplicate_predictions = (
    pred_df
    .duplicated(
        subset=[
            "exp_name",
            "target_frame",
        ],
        keep=False,
    )
)


print(
    "\nDuplicate prediction "
    "(experiment, target_frame) rows:",
    int(
        duplicate_predictions.sum()
    )
)


# ==================================================================================================
# 18. STRICT H=10 TARGET-FRAME MATCHING
# ==================================================================================================

print("\n" + "=" * 120)
print("STRICT H=10 PREDICTION / COCO TARGET-FRAME MATCHING")
print("=" * 120)


# IMPORTANT:
#
# Prediction at t predicts target_frame = t+10
#
# We therefore match:
#
# pred.target_frame
#
# to
#
# COCO.frame_idx


merged = pred_df.merge(

    gt_df,

    left_on=[
        "exp_name",
        "target_frame",
    ],

    right_on=[
        "exp_name",
        "frame_idx",
    ],

    how="inner",

    suffixes=(
        "_prediction",
        "_gt",
    ),
)


print(
    f"\nTotal H=10 predictions      : {len(pred_df)}"
)

print(
    f"Total COCO annotated frames : {len(gt_df)}"
)

print(
    f"Matched prediction-GT pairs : {len(merged)}"
)


# ==================================================================================================
# 19. MATCHING SUMMARY BY EXPERIMENT
# ==================================================================================================

match_summary_rows = []


for exp_name in PRED_FILES.keys():

    prediction_exp = pred_df[
        pred_df[
            "exp_name"
        ]
        == exp_name
    ]


    gt_exp = gt_df[
        gt_df[
            "exp_name"
        ]
        == exp_name
    ]


    merged_exp = merged[
        merged[
            "exp_name"
        ]
        == exp_name
    ]


    match_summary_rows.append({
        "experiment":
            exp_name,
        "H10_prediction_rows":
            len(
                prediction_exp
            ),
        "COCO_annotated_frames":
            len(
                gt_exp
            ),
        "matched_H10_samples":
            len(
                merged_exp
            ),
    })


match_summary_df = pd.DataFrame(
    match_summary_rows
)


print("\nMatching summary:")

print(
    match_summary_df.to_string(
        index=False
    )
)


match_summary_df.to_csv(
    OUTPUT_DIR
    / "H10_matching_summary.csv",
    index=False,
)


# ==================================================================================================
# 20. EXCLUDE ONLY UNRESOLVED GT FRAMES
# ==================================================================================================

ambiguous_count = int(
    merged[
        "gt_class"
    ]
    .isna()
    .sum()
)


print(
    "\nMatched frames without a unique GT class:",
    ambiguous_count
)


evaluation_df = merged[
    merged[
        "gt_class"
    ]
    .notna()
].copy()


print(
    "\nFinal valid multiclass evaluation samples:",
    len(
        evaluation_df
    )
)


if len(
    evaluation_df
) == 0:

    raise RuntimeError(
        "\nNo valid samples remain after GT audit."
    )


# ==================================================================================================
# 21. GROUND-TRUTH DISTRIBUTION
# ==================================================================================================

print("\n" + "=" * 120)
print("FINAL MATCHED GROUND-TRUTH DISTRIBUTION")
print("=" * 120)


distribution = (
    evaluation_df
    .groupby(
        [
            "exp_name",
            "gt_class",
        ]
    )
    .size()
)


print(
    distribution
)


distribution.to_csv(
    OUTPUT_DIR
    / "H10_matched_ground_truth_distribution.csv"
)


# ==================================================================================================
# 22. SAVE COMPLETE MATCHED DATASET
# ==================================================================================================

matched_file = (
    OUTPUT_DIR
    / "H10_ZERO_SHOT_matched_predictions_ground_truth.csv"
)


evaluation_df.to_csv(
    matched_file,
    index=False,
)


print(
    f"\nMatched evaluation dataset saved:\n"
    f"{matched_file}"
)


# ==================================================================================================
# 23. PER-EXPERIMENT MULTICLASS PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("PER-EXPERIMENT H=10 ZERO-SHOT MULTICLASS PERFORMANCE")
print("=" * 120)


per_experiment_rows = []


for exp_name in PRED_FILES.keys():

    temp = evaluation_df[
        evaluation_df[
            "exp_name"
        ]
        == exp_name
    ]


    if len(
        temp
    ) == 0:

        print(
            f"\nWARNING: "
            f"{exp_name} has no valid matched samples."
        )

        continue


    y_true_exp = temp[
        "gt_class"
    ]

    y_pred_exp = temp[
        "predicted_class"
    ]


    metrics = calculate_metrics(
        y_true_exp,
        y_pred_exp,
    )


    row = {
        "experiment":
            exp_name,
        "samples":
            len(
                temp
            ),
        "accuracy_percent":
            metrics[
                "accuracy"
            ] * 100,
        "macro_precision_percent":
            metrics[
                "macro_precision"
            ] * 100,
        "macro_recall_percent":
            metrics[
                "macro_recall"
            ] * 100,
        "macro_f1_percent":
            metrics[
                "macro_f1"
            ] * 100,
        "weighted_f1_percent":
            metrics[
                "weighted_f1"
            ] * 100,
    }


    per_experiment_rows.append(
        row
    )


    print(
        f"\n{exp_name.upper()}"
    )

    print(
        f"  Samples         : {len(temp)}"
    )

    print(
        f"  Accuracy        : "
        f"{metrics['accuracy'] * 100:.2f}%"
    )

    print(
        f"  Macro Precision : "
        f"{metrics['macro_precision'] * 100:.2f}%"
    )

    print(
        f"  Macro Recall    : "
        f"{metrics['macro_recall'] * 100:.2f}%"
    )

    print(
        f"  Macro F1        : "
        f"{metrics['macro_f1'] * 100:.2f}%"
    )

    print(
        f"  Weighted F1     : "
        f"{metrics['weighted_f1'] * 100:.2f}%"
    )


per_experiment_df = pd.DataFrame(
    per_experiment_rows
)


per_experiment_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_per_experiment_multiclass_metrics.csv",
    index=False,
)


# ==================================================================================================
# 24. POOLED MULTICLASS PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("POOLED H=10 ZERO-SHOT MULTICLASS PERFORMANCE")
print("=" * 120)


y_true = evaluation_df[
    "gt_class"
]

y_pred = evaluation_df[
    "predicted_class"
]


pooled = calculate_metrics(
    y_true,
    y_pred,
)


print(
    f"\nEvaluation samples : {len(evaluation_df)}"
)

print(
    f"Accuracy            : "
    f"{pooled['accuracy'] * 100:.2f}%"
)

print(
    f"Macro Precision     : "
    f"{pooled['macro_precision'] * 100:.2f}%"
)

print(
    f"Macro Recall        : "
    f"{pooled['macro_recall'] * 100:.2f}%"
)

print(
    f"Macro F1            : "
    f"{pooled['macro_f1'] * 100:.2f}%"
)

print(
    f"Weighted F1         : "
    f"{pooled['weighted_f1'] * 100:.2f}%"
)


pooled_df = pd.DataFrame(
    [{
        "samples":
            len(
                evaluation_df
            ),
        "accuracy_percent":
            pooled[
                "accuracy"
            ] * 100,
        "macro_precision_percent":
            pooled[
                "macro_precision"
            ] * 100,
        "macro_recall_percent":
            pooled[
                "macro_recall"
            ] * 100,
        "macro_f1_percent":
            pooled[
                "macro_f1"
            ] * 100,
        "weighted_f1_percent":
            pooled[
                "weighted_f1"
            ] * 100,
    }]
)


pooled_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_pooled_multiclass_metrics.csv",
    index=False,
)


# ==================================================================================================
# 25. MULTICLASS CLASSIFICATION REPORT
# ==================================================================================================

print("\n" + "=" * 120)
print("POOLED MULTICLASS CLASSIFICATION REPORT")
print("=" * 120)


classification = classification_report(

    y_true,
    y_pred,

    labels=CLASS_ORDER,

    target_names=CLASS_ORDER,

    zero_division=0,

    output_dict=True,
)


classification_df = pd.DataFrame(
    classification
).transpose()


print(
    classification_df
    .round(
        4
    )
    .to_string()
)


classification_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_multiclass_classification_report.csv"
)


# ==================================================================================================
# 26. MULTICLASS CONFUSION MATRIX
# ==================================================================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=CLASS_ORDER,
)


cm_df = pd.DataFrame(
    cm,
    index=CLASS_ORDER,
    columns=CLASS_ORDER,
)


print("\n" + "=" * 120)
print("POOLED MULTICLASS CONFUSION MATRIX")
print("=" * 120)


print(
    "\nClass order:"
)

print(
    CLASS_ORDER
)


print(
    "\nRaw confusion matrix:"
)

print(
    cm_df
)


cm_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_multiclass_confusion_matrix_RAW.csv"
)


# ==================================================================================================
# 27. NORMALIZED MULTICLASS CONFUSION MATRIX
# ==================================================================================================

row_sums = cm.sum(
    axis=1,
    keepdims=True,
)


cm_norm = np.divide(

    cm,

    row_sums,

    out=np.zeros_like(
        cm,
        dtype=float,
    ),

    where=(
        row_sums != 0
    ),

) * 100


cm_norm_df = pd.DataFrame(
    cm_norm,
    index=CLASS_ORDER,
    columns=CLASS_ORDER,
)


print(
    "\nRow-normalized confusion matrix (%):"
)

print(
    cm_norm_df.round(
        2
    )
)


cm_norm_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_multiclass_confusion_matrix_NORMALIZED.csv"
)


# ==================================================================================================
# 28. PLOT MULTICLASS CONFUSION MATRIX
# ==================================================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        7,
    )
)


ax.imshow(
    cm_norm
)


ax.set_xticks(
    np.arange(
        len(
            CLASS_ORDER
        )
    )
)

ax.set_yticks(
    np.arange(
        len(
            CLASS_ORDER
        )
    )
)


ax.set_xticklabels(
    CLASS_ORDER,
    rotation=30,
    ha="right",
)

ax.set_yticklabels(
    CLASS_ORDER
)


for row in range(
    len(
        CLASS_ORDER
    )
):

    for col in range(
        len(
            CLASS_ORDER
        )
    ):

        ax.text(

            col,
            row,

            (
                f"{cm_norm[row, col]:.1f}%\n"
                f"(n={cm[row, col]})"
            ),

            ha="center",
            va="center",
        )


ax.set_xlabel(
    "Predicted future weld-quality class"
)

ax.set_ylabel(
    "Ground-truth future weld-quality class"
)

ax.set_title(
    "H=10 Zero-Shot Cross-Experiment Evaluation"
)


fig.tight_layout()


fig.savefig(

    OUTPUT_DIR
    / "H10_zero_shot_multiclass_confusion_matrix.png",

    dpi=300,
    bbox_inches="tight",
)


plt.show()


# ==================================================================================================
# 29. BINARY GOOD vs DEFECT LABELS
# ==================================================================================================

evaluation_df[
    "gt_binary"
] = np.where(

    evaluation_df[
        "gt_class"
    ]
    == "Good",

    "Good",

    "Defect",
)


evaluation_df[
    "pred_binary"
] = np.where(

    evaluation_df[
        "predicted_class"
    ]
    == "Good",

    "Good",

    "Defect",
)


# ==================================================================================================
# 30. POOLED BINARY PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("POOLED H=10 ZERO-SHOT BINARY FUTURE WELD-STATE PERFORMANCE")
print("=" * 120)


binary_true = evaluation_df[
    "gt_binary"
]

binary_pred = evaluation_df[
    "pred_binary"
]


binary_metrics = calculate_metrics(
    binary_true,
    binary_pred,
)


print(
    f"\nAccuracy        : "
    f"{binary_metrics['accuracy'] * 100:.2f}%"
)

print(
    f"Macro Precision : "
    f"{binary_metrics['macro_precision'] * 100:.2f}%"
)

print(
    f"Macro Recall    : "
    f"{binary_metrics['macro_recall'] * 100:.2f}%"
)

print(
    f"Macro F1        : "
    f"{binary_metrics['macro_f1'] * 100:.2f}%"
)

print(
    f"Weighted F1     : "
    f"{binary_metrics['weighted_f1'] * 100:.2f}%"
)


# ==================================================================================================
# 31. BINARY CLASSIFICATION REPORT
# ==================================================================================================

binary_report = classification_report(

    binary_true,
    binary_pred,

    labels=BINARY_ORDER,

    target_names=BINARY_ORDER,

    zero_division=0,

    output_dict=True,
)


binary_report_df = pd.DataFrame(
    binary_report
).transpose()


print(
    "\nBinary classification report:"
)

print(
    binary_report_df
    .round(
        4
    )
    .to_string()
)


binary_report_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_binary_classification_report.csv"
)


# ==================================================================================================
# 32. BINARY CONFUSION MATRIX
# ==================================================================================================

binary_cm = confusion_matrix(

    binary_true,
    binary_pred,

    labels=BINARY_ORDER,
)


binary_cm_df = pd.DataFrame(
    binary_cm,
    index=BINARY_ORDER,
    columns=BINARY_ORDER,
)


print(
    "\nBinary confusion matrix:"
)

print(
    binary_cm_df
)


binary_cm_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_binary_confusion_matrix_RAW.csv"
)


# ==================================================================================================
# 33. NORMALIZED BINARY CONFUSION MATRIX
# ==================================================================================================

binary_row_sums = binary_cm.sum(
    axis=1,
    keepdims=True,
)


binary_cm_norm = np.divide(

    binary_cm,

    binary_row_sums,

    out=np.zeros_like(
        binary_cm,
        dtype=float,
    ),

    where=(
        binary_row_sums != 0
    ),

) * 100


binary_cm_norm_df = pd.DataFrame(
    binary_cm_norm,
    index=BINARY_ORDER,
    columns=BINARY_ORDER,
)


print(
    "\nNormalized binary confusion matrix (%):"
)

print(
    binary_cm_norm_df.round(
        2
    )
)


binary_cm_norm_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_binary_confusion_matrix_NORMALIZED.csv"
)


# ==================================================================================================
# 34. PLOT BINARY CONFUSION MATRIX
# ==================================================================================================

fig, ax = plt.subplots(
    figsize=(
        6,
        5,
    )
)


ax.imshow(
    binary_cm_norm
)


ax.set_xticks(
    np.arange(
        len(
            BINARY_ORDER
        )
    )
)

ax.set_yticks(
    np.arange(
        len(
            BINARY_ORDER
        )
    )
)


ax.set_xticklabels(
    BINARY_ORDER
)

ax.set_yticklabels(
    BINARY_ORDER
)


for row in range(
    len(
        BINARY_ORDER
    )
):

    for col in range(
        len(
            BINARY_ORDER
        )
    ):

        ax.text(

            col,
            row,

            (
                f"{binary_cm_norm[row, col]:.1f}%\n"
                f"(n={binary_cm[row, col]})"
            ),

            ha="center",
            va="center",
        )


ax.set_xlabel(
    "Predicted future weld state"
)

ax.set_ylabel(
    "Ground-truth future weld state"
)

ax.set_title(
    "H=10 Zero-Shot Binary Future Weld-State Evaluation"
)


fig.tight_layout()


fig.savefig(

    OUTPUT_DIR
    / "H10_zero_shot_binary_confusion_matrix.png",

    dpi=300,
    bbox_inches="tight",
)


plt.show()


# ==================================================================================================
# 35. PER-EXPERIMENT BINARY PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("PER-EXPERIMENT H=10 ZERO-SHOT BINARY PERFORMANCE")
print("=" * 120)


binary_per_exp_rows = []


for exp_name in PRED_FILES.keys():

    temp = evaluation_df[
        evaluation_df[
            "exp_name"
        ]
        == exp_name
    ]


    if len(
        temp
    ) == 0:

        continue


    metrics = calculate_metrics(

        temp[
            "gt_binary"
        ],

        temp[
            "pred_binary"
        ],
    )


    binary_per_exp_rows.append({

        "experiment":
            exp_name,

        "samples":
            len(
                temp
            ),

        "accuracy_percent":
            metrics[
                "accuracy"
            ] * 100,

        "macro_precision_percent":
            metrics[
                "macro_precision"
            ] * 100,

        "macro_recall_percent":
            metrics[
                "macro_recall"
            ] * 100,

        "macro_f1_percent":
            metrics[
                "macro_f1"
            ] * 100,

        "weighted_f1_percent":
            metrics[
                "weighted_f1"
            ] * 100,
    })


binary_per_exp_df = pd.DataFrame(
    binary_per_exp_rows
)


print(
    binary_per_exp_df
    .round(
        2
    )
    .to_string(
        index=False
    )
)


binary_per_exp_df.to_csv(
    OUTPUT_DIR
    / "H10_zero_shot_per_experiment_binary_metrics.csv",
    index=False,
)


# ==================================================================================================
# 36. SAVE FINAL EVALUATION DATASET
# ==================================================================================================

evaluation_df.to_csv(

    OUTPUT_DIR
    / "H10_ZERO_SHOT_FINAL_EVALUATION_DATASET.csv",

    index=False,
)


# ==================================================================================================
# 37. MANUSCRIPT-READY SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("MANUSCRIPT-READY H=10 ZERO-SHOT VALIDATION SUMMARY")
print("=" * 120)


print(
    f"""
Independent experiments : Exp. 12, Exp. 13, Exp. 14
Forecasting horizon     : H=10
Nominal lead time       : ~0.333 s
Valid matched samples   : {len(evaluation_df)}

MULTICLASS ZERO-SHOT PERFORMANCE
Accuracy                : {pooled['accuracy'] * 100:.2f}%
Macro Precision         : {pooled['macro_precision'] * 100:.2f}%
Macro Recall            : {pooled['macro_recall'] * 100:.2f}%
Macro F1                : {pooled['macro_f1'] * 100:.2f}%
Weighted F1             : {pooled['weighted_f1'] * 100:.2f}%

BINARY GOOD-vs-DEFECT ZERO-SHOT PERFORMANCE
Accuracy                : {binary_metrics['accuracy'] * 100:.2f}%
Macro Precision         : {binary_metrics['macro_precision'] * 100:.2f}%
Macro Recall            : {binary_metrics['macro_recall'] * 100:.2f}%
Macro F1                : {binary_metrics['macro_f1'] * 100:.2f}%
Weighted F1             : {binary_metrics['weighted_f1'] * 100:.2f}%
"""
)


# ==================================================================================================
# 38. FINAL AUDIT MESSAGE
# ==================================================================================================

print("=" * 120)
print("FINAL H=10 ZERO-SHOT CROSS-EXPERIMENT EVALUATION COMPLETE")
print("=" * 120)


print(
    f"\nResults saved to:\n{OUTPUT_DIR}"
)


print(
    "\nBEFORE USING THESE VALUES IN THE MANUSCRIPT, CHECK:"
)

print(
    "  1. COCO category mapping"
)

print(
    "  2. Annotation counts for Exp. 12, 13, and 14"
)

print(
    "  3. Filename parse failures"
)

print(
    "  4. Unknown-category frames"
)

print(
    "  5. Multi-defect frames"
)

print(
    "  6. Matching summary"
)

print(
    "  7. Final GT class distribution"
)

print(
    "  8. Per-experiment metrics"
)

print(
    "  9. Pooled confusion matrix"
)


print("\nDone.")

In [ ]:
# ==================================================================================================
# FINAL CORRECTED H=10 ZERO-SHOT CROSS-EXPERIMENT EVALUATION
# Exp. 12, Exp. 13, Exp. 14
#
# Corrected:
#   1. Robust Roboflow filename parsing
#   2. Correct flash_bur COCO mapping
#   3. Dominant annotated defect area used for multiclass GT
#   4. Good assigned only when no recognized defect is annotated
#   5. Prediction target_frame matched directly to independently annotated frame
#   6. Strict 896-frame and duplicate audits
# ==================================================================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)


# ==================================================================================================
# 1. PATHS
# ==================================================================================================

PROJECT_DIR = Path.cwd()

COCO_JSON = Path(
    "data/external/_annotations.coco.json"
)

PREDICTION_DIR = (
    PROJECT_DIR
    / "external_test"
    / "external_test_outputs_H10_full_features_FINAL"
)

PREDICTION_FILES = {
    "exp_12": PREDICTION_DIR / "exp_12_H10_predictions_full_features.csv",
    "exp_13": PREDICTION_DIR / "exp_13_H10_predictions_full_features.csv",
    "exp_14": PREDICTION_DIR / "exp_14_H10_predictions_full_features.csv",
}

OUTPUT_DIR = (
    PROJECT_DIR
    / "external_test"
    / "FINAL_H10_ZERO_SHOT_COCO_EVALUATION_CORRECTED"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================================================
# 2. SETTINGS
# ==================================================================================================

HORIZON = 10

EXPERIMENTS = [
    "exp_12",
    "exp_13",
    "exp_14",
]

CLASS_ID_TO_NAME = {
    0: "Good",
    1: "Burr",
    2: "Flash-burr",
    3: "Surface-groove/void",
}

CLASS_NAMES = [
    "Good",
    "Burr",
    "Flash-burr",
    "Surface-groove/void",
]

CLASS_IDS = [0, 1, 2, 3]


# ==================================================================================================
# 3. COCO CATEGORY MAPPING
# ==================================================================================================

CATEGORY_ALIASES = {

    # Burr
    "burr": 1,
    "burrs": 1,

    # Flash-burr
    "flash_bur": 2,       # <-- IMPORTANT: actual COCO name
    "flash_burr": 2,
    "flashburr": 2,
    "flash_burrs": 2,

    # Surface groove / void
    "surface_groove_void": 3,
    "surface_groove": 3,
    "surface_void": 3,
    "surface_groove_voids": 3,
}


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def normalize_text(value):

    text = str(value).strip().lower()

    text = re.sub(
        r"[\s\-/]+",
        "_",
        text,
    )

    text = re.sub(
        r"_+",
        "_",
        text,
    )

    return text.strip("_")


def parse_roboflow_filename(filename):
    """
    Correctly parse filenames such as:

        Exp_12_000207_jpg.rf.cb1e495e....jpg
        Exp_13_000298_jpg.rf.9c36fe7....jpg
        EXP_14_000375_jpg.rf.7305983....jpg

    Returns:
        exp_12, 207
        exp_13, 298
        exp_14, 375

    IMPORTANT:
    The frame number is taken ONLY from the numeric token
    immediately following the experiment number.
    The Roboflow hash is ignored completely.
    """

    name = Path(
        str(filename)
    ).name

    match = re.search(
        r"(?i)exp[_\-]?(\d+)[_\-](\d+)",
        name,
    )

    if match is None:
        return None, None

    exp_number = int(
        match.group(1)
    )

    frame_number = int(
        match.group(2)
    )

    return (
        f"exp_{exp_number}",
        frame_number,
    )


def annotation_area(annotation):
    """
    Obtain independent annotation area.

    Priority:
      1. COCO area field
      2. Polygon area
      3. Bounding-box area
    """

    area = annotation.get(
        "area",
        None,
    )

    if area is not None:

        try:

            area = float(area)

            if np.isfinite(area) and area >= 0:
                return area

        except Exception:
            pass


    segmentation = annotation.get(
        "segmentation",
        None,
    )

    if isinstance(segmentation, list):

        total = 0.0

        for polygon in segmentation:

            if not isinstance(
                polygon,
                list,
            ):
                continue

            coords = np.asarray(
                polygon,
                dtype=float,
            )

            if (
                len(coords) < 6
                or len(coords) % 2 != 0
            ):
                continue

            x = coords[0::2]
            y = coords[1::2]

            poly_area = 0.5 * abs(
                np.dot(
                    x,
                    np.roll(y, 1),
                )
                -
                np.dot(
                    y,
                    np.roll(x, 1),
                )
            )

            total += poly_area

        if total > 0:
            return total


    bbox = annotation.get(
        "bbox",
        None,
    )

    if (
        bbox is not None
        and len(bbox) >= 4
    ):

        try:

            return max(
                0.0,
                float(bbox[2])
                * float(bbox[3]),
            )

        except Exception:
            pass


    return 0.0


def calculate_metrics(
    y_true,
    y_pred,
    labels,
):

    return {

        "n_samples":
            len(y_true),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                labels=labels,
                average="weighted",
                zero_division=0,
            ),
    }


def plot_confusion_matrix(
    cm,
    class_names,
    title,
    output_file,
):

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )

    im = ax.imshow(cm)

    fig.colorbar(
        im,
        ax=ax,
    )

    ax.set_xticks(
        np.arange(
            len(class_names)
        )
    )

    ax.set_yticks(
        np.arange(
            len(class_names)
        )
    )

    ax.set_xticklabels(
        class_names,
        rotation=30,
        ha="right",
    )

    ax.set_yticklabels(
        class_names
    )

    ax.set_xlabel(
        "Predicted class"
    )

    ax.set_ylabel(
        "True class"
    )

    ax.set_title(
        title
    )

    for i in range(
        cm.shape[0]
    ):

        for j in range(
            cm.shape[1]
        ):

            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
            )

    fig.tight_layout()

    fig.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


# ==================================================================================================
# 5. LOAD COCO
# ==================================================================================================

print("=" * 120)
print("FINAL CORRECTED H=10 ZERO-SHOT CROSS-EXPERIMENT EVALUATION")
print("=" * 120)

assert COCO_JSON.exists(), (
    f"COCO JSON not found:\n{COCO_JSON}"
)

for exp, path in PREDICTION_FILES.items():

    assert path.exists(), (
        f"Prediction file missing:\n{path}"
    )


with open(
    COCO_JSON,
    "r",
    encoding="utf-8",
) as file:

    coco = json.load(file)


images = coco.get(
    "images",
    [],
)

annotations = coco.get(
    "annotations",
    [],
)

categories = coco.get(
    "categories",
    [],
)


print("\nCOCO:")
print("Images      :", len(images))
print("Annotations :", len(annotations))
print("Categories  :", len(categories))


# ==================================================================================================
# 6. CATEGORY AUDIT
# ==================================================================================================

category_id_to_name = {
    int(category["id"]):
        str(category["name"])
    for category in categories
}


category_counts = {}

for annotation in annotations:

    category_id = int(
        annotation["category_id"]
    )

    category_counts[
        category_id
    ] = (
        category_counts.get(
            category_id,
            0,
        )
        + 1
    )


print("\n" + "=" * 120)
print("COCO CATEGORY AUDIT")
print("=" * 120)


category_rows = []

for category in categories:

    category_id = int(
        category["id"]
    )

    category_name = str(
        category["name"]
    )

    normalized = normalize_text(
        category_name
    )

    mapped_id = CATEGORY_ALIASES.get(
        normalized,
        None,
    )

    mapped_name = (
        CLASS_ID_TO_NAME[
            mapped_id
        ]
        if mapped_id is not None
        else "IGNORED"
    )

    row = {
        "category_id":
            category_id,

        "category_name":
            category_name,

        "normalized":
            normalized,

        "annotation_count":
            category_counts.get(
                category_id,
                0,
            ),

        "mapped_class_id":
            mapped_id,

        "mapped_class":
            mapped_name,
    }

    category_rows.append(
        row
    )


category_df = pd.DataFrame(
    category_rows
)


print(
    category_df.to_string(
        index=False
    )
)


category_df.to_csv(
    OUTPUT_DIR
    / "COCO_category_audit.csv",
    index=False,
)


# ==================================================================================================
# 7. CORRECT IMAGE/FILENAME PARSING
# ==================================================================================================

print("\n" + "=" * 120)
print("ROBUST ROBOFLOW FILENAME PARSING")
print("=" * 120)


image_rows = []

parse_failures = []


for image in images:

    image_id = int(
        image["id"]
    )

    file_name = str(
        image["file_name"]
    )

    exp_id, frame_idx = (
        parse_roboflow_filename(
            file_name
        )
    )

    if (
        exp_id is None
        or frame_idx is None
    ):

        parse_failures.append(
            file_name
        )

        continue


    image_rows.append(
        {
            "image_id":
                image_id,

            "file_name":
                file_name,

            "exp_id":
                exp_id,

            "annotation_frame_idx":
                frame_idx,
        }
    )


image_df = pd.DataFrame(
    image_rows
)


print("\nFilename parse failures:")
print(
    len(parse_failures)
)


print("\nFirst 20 correctly parsed filenames:")

print(
    image_df
    .head(20)
    .to_string(
        index=False
    )
)


print("\nFrame ranges by experiment:")

print(
    image_df
    .groupby(
        "exp_id"
    )[
        "annotation_frame_idx"
    ]
    .agg(
        [
            "min",
            "max",
            "count",
        ]
    )
)


# ==================================================================================================
# 8. CRITICAL EXPECTED FRAME-RANGE AUDIT
# ==================================================================================================

expected_counts = {
    "exp_12": 298,
    "exp_13": 349,
    "exp_14": 249,
}


print("\n" + "=" * 120)
print("EXPECTED ANNOTATION COUNT AUDIT")
print("=" * 120)


for exp_id, expected_count in expected_counts.items():

    actual = int(
        (
            image_df["exp_id"]
            == exp_id
        ).sum()
    )

    print(
        f"{exp_id}: "
        f"{actual} / {expected_count}"
    )

    if actual != expected_count:

        raise RuntimeError(
            f"\n{exp_id} annotation-count audit FAILED."
        )


if len(image_df) != 896:

    raise RuntimeError(
        f"\nExpected 896 parsed COCO images, "
        f"found {len(image_df)}."
    )


print(
    "\nPASS: all 896 COCO filenames parsed correctly."
)


# ==================================================================================================
# 9. INITIALIZE GROUND-TRUTH AREA TABLE
# ==================================================================================================

gt_df = image_df.copy()


gt_df[
    "Burr_annotation_area"
] = 0.0

gt_df[
    "Flash_burr_annotation_area"
] = 0.0

gt_df[
    "Surface_groove_void_annotation_area"
] = 0.0


image_id_to_index = {
    image_id: index
    for index, image_id
    in enumerate(
        gt_df["image_id"]
    )
}


ignored_counter = {}


# ==================================================================================================
# 10. ACCUMULATE INDEPENDENT COCO DEFECT AREAS
# ==================================================================================================

for annotation in annotations:

    image_id = int(
        annotation["image_id"]
    )

    if image_id not in image_id_to_index:
        continue


    category_id = int(
        annotation["category_id"]
    )

    category_name = (
        category_id_to_name[
            category_id
        ]
    )

    normalized = normalize_text(
        category_name
    )

    class_id = CATEGORY_ALIASES.get(
        normalized,
        None,
    )


    if class_id is None:

        ignored_counter[
            category_name
        ] = (
            ignored_counter.get(
                category_name,
                0,
            )
            + 1
        )

        continue


    area = annotation_area(
        annotation
    )


    index = image_id_to_index[
        image_id
    ]


    if class_id == 1:

        gt_df.loc[
            index,
            "Burr_annotation_area",
        ] += area


    elif class_id == 2:

        gt_df.loc[
            index,
            "Flash_burr_annotation_area",
        ] += area


    elif class_id == 3:

        gt_df.loc[
            index,
            "Surface_groove_void_annotation_area",
        ] += area


print("\nIgnored categories:")

for name, count in ignored_counter.items():

    print(
        f"  {name}: {count}"
    )


# ==================================================================================================
# 11. ASSIGN DOMINANT-AREA MULTICLASS GROUND TRUTH
# ==================================================================================================

def assign_gt(row):

    areas = {

        1:
            float(
                row[
                    "Burr_annotation_area"
                ]
            ),

        2:
            float(
                row[
                    "Flash_burr_annotation_area"
                ]
            ),

        3:
            float(
                row[
                    "Surface_groove_void_annotation_area"
                ]
            ),
    }


    largest_area = max(
        areas.values()
    )


    # No recognized defect annotation
    if largest_area <= 0:
        return 0


    winners = [
        class_id
        for class_id, area
        in areas.items()
        if np.isclose(
            area,
            largest_area,
            atol=1e-12,
            rtol=0,
        )
    ]


    if len(winners) != 1:
        return -1


    return winners[0]


gt_df[
    "true_class_id"
] = gt_df.apply(
    assign_gt,
    axis=1,
)


ties = gt_df[
    gt_df["true_class_id"]
    == -1
]


if len(ties) > 0:

    tie_file = (
        OUTPUT_DIR
        / "WARNING_dominant_area_ties.csv"
    )

    ties.to_csv(
        tie_file,
        index=False,
    )

    raise RuntimeError(
        f"\nDominant-area ties detected: {len(ties)}\n"
        f"Inspect:\n{tie_file}"
    )


gt_df[
    "true_class"
] = gt_df[
    "true_class_id"
].map(
    CLASS_ID_TO_NAME
)


gt_df[
    "binary_true"
] = np.where(
    gt_df[
        "true_class_id"
    ] == 0,
    "Good",
    "Defect",
)


# ==================================================================================================
# 12. FINAL GT AUDIT
# ==================================================================================================

print("\n" + "=" * 120)
print("FINAL INDEPENDENT COCO GROUND-TRUTH AUDIT")
print("=" * 120)


print("\nOverall GT distribution:")

print(
    gt_df[
        "true_class"
    ]
    .value_counts()
)


print("\nGT distribution by experiment:")

print(
    pd.crosstab(
        gt_df[
            "exp_id"
        ],
        gt_df[
            "true_class"
        ],
        margins=True,
    )
)


gt_df.to_csv(
    OUTPUT_DIR
    / "FINAL_independent_COCO_ground_truth.csv",
    index=False,
)


# ==================================================================================================
# 13. LOAD FINAL H=10 PREDICTIONS
# ==================================================================================================

prediction_tables = []


for exp_id, path in PREDICTION_FILES.items():

    df = pd.read_csv(
        path
    )


    # Your files use exp_name
    if "exp_name" in df.columns:

        df["exp_id"] = (
            df["exp_name"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    else:

        df["exp_id"] = exp_id


    df[
        "frame_idx"
    ] = pd.to_numeric(
        df["frame_idx"]
    ).astype(int)


    df[
        "target_frame"
    ] = pd.to_numeric(
        df["target_frame"]
    ).astype(int)


    offsets = sorted(
        (
            df[
                "target_frame"
            ]
            -
            df[
                "frame_idx"
            ]
        )
        .unique()
        .tolist()
    )


    print(
        f"\n{exp_id}: horizon offsets = {offsets}"
    )


    if offsets != [10]:

        raise RuntimeError(
            f"{exp_id} is not pure H=10."
        )


    if "predicted_class_id" not in df.columns:

        raise RuntimeError(
            f"'predicted_class_id' missing in {path}"
        )


    df[
        "pred_class_id"
    ] = pd.to_numeric(
        df[
            "predicted_class_id"
        ]
    ).astype(int)


    df[
        "pred_class"
    ] = df[
        "pred_class_id"
    ].map(
        CLASS_ID_TO_NAME
    )


    prediction_tables.append(
        df
    )


pred_df = pd.concat(
    prediction_tables,
    ignore_index=True,
)


# ==================================================================================================
# 14. STRICT MATCHING
# ==================================================================================================

print("\n" + "=" * 120)
print("STRICT H=10 TARGET FRAME MATCHING")
print("=" * 120)


evaluation_df = pred_df.merge(

    gt_df,

    left_on=[
        "exp_id",
        "target_frame",
    ],

    right_on=[
        "exp_id",
        "annotation_frame_idx",
    ],

    how="inner",

    suffixes=(
        "_prediction",
        "_GT",
    ),
)


print(
    "\nMatched rows:",
    len(evaluation_df)
)


print("\nMatched rows by experiment:")

print(
    evaluation_df[
        "exp_id"
    ]
    .value_counts()
    .sort_index()
)


# ==================================================================================================
# 15. CRITICAL 896-SAMPLE CHECK
# ==================================================================================================

if len(evaluation_df) != 896:

    raise RuntimeError(
        "\nFINAL MATCHING AUDIT FAILED.\n"
        f"Expected 896 matched samples, got {len(evaluation_df)}.\n"
        "Do not use metrics."
    )


print(
    "\nPASS: all 896 expected annotated samples were matched."
)


# ==================================================================================================
# 16. DUPLICATE AUDIT
# ==================================================================================================

duplicates = evaluation_df.duplicated(

    subset=[
        "exp_id",
        "target_frame",
    ],

    keep=False,
)


duplicate_count = int(
    duplicates.sum()
)


print(
    "\nDuplicate matched targets:",
    duplicate_count
)


if duplicate_count != 0:

    duplicate_file = (
        OUTPUT_DIR
        / "WARNING_duplicate_matches.csv"
    )

    evaluation_df[
        duplicates
    ].to_csv(
        duplicate_file,
        index=False,
    )

    raise RuntimeError(
        "\nDuplicate target-frame matches detected.\n"
        f"Inspect:\n{duplicate_file}"
    )


print(
    "PASS: no duplicate target-frame matches."
)


# ==================================================================================================
# 17. SAVE MATCHED DATASET
# ==================================================================================================

evaluation_df.to_csv(

    OUTPUT_DIR
    / "FINAL_H10_ZERO_SHOT_MATCHED_896.csv",

    index=False,
)


# ==================================================================================================
# 18. POOLED MULTICLASS METRICS
# ==================================================================================================

y_true = evaluation_df[
    "true_class_id"
].astype(int)

y_pred = evaluation_df[
    "pred_class_id"
].astype(int)


multiclass_metrics = calculate_metrics(

    y_true,
    y_pred,
    labels=CLASS_IDS,
)


print("\n" + "=" * 120)
print("POOLED H=10 ZERO-SHOT MULTICLASS PERFORMANCE")
print("=" * 120)


for key, value in multiclass_metrics.items():

    if key == "n_samples":

        print(
            f"{key:25s}: {value}"
        )

    else:

        print(
            f"{key:25s}: {value * 100:.2f}%"
        )


print("\nClassification report:")

print(
    classification_report(

        y_true,
        y_pred,

        labels=CLASS_IDS,

        target_names=CLASS_NAMES,

        zero_division=0,

        digits=4,
    )
)


# ==================================================================================================
# 19. POOLED MULTICLASS CONFUSION MATRIX
# ==================================================================================================

multiclass_cm = confusion_matrix(

    y_true,
    y_pred,

    labels=CLASS_IDS,
)


print(
    "\nRaw multiclass confusion matrix:"
)

print(
    multiclass_cm
)


multiclass_cm_df = pd.DataFrame(

    multiclass_cm,

    index=CLASS_NAMES,

    columns=CLASS_NAMES,
)


multiclass_cm_df.to_csv(

    OUTPUT_DIR
    / "pooled_multiclass_confusion_matrix_RAW.csv"
)


plot_confusion_matrix(

    multiclass_cm,

    CLASS_NAMES,

    "Independent H=10 zero-shot multiclass evaluation — Exp. 12–14",

    OUTPUT_DIR
    / "pooled_multiclass_confusion_matrix.png",
)


# ==================================================================================================
# 20. NORMALIZED MULTICLASS CONFUSION MATRIX
# ==================================================================================================

multiclass_row_sums = multiclass_cm.sum(
    axis=1,
    keepdims=True,
)


multiclass_cm_normalized = np.divide(

    multiclass_cm,

    multiclass_row_sums,

    out=np.zeros_like(
        multiclass_cm,
        dtype=float,
    ),

    where=(
        multiclass_row_sums != 0
    ),

) * 100


print(
    "\nNormalized multiclass confusion matrix (%):"
)

print(
    np.round(
        multiclass_cm_normalized,
        2,
    )
)


pd.DataFrame(

    multiclass_cm_normalized,

    index=CLASS_NAMES,

    columns=CLASS_NAMES,

).to_csv(

    OUTPUT_DIR
    / "pooled_multiclass_confusion_matrix_NORMALIZED.csv"
)


# ==================================================================================================
# 21. POOLED BINARY GOOD vs DEFECT
# ==================================================================================================

binary_true = (
    y_true != 0
).astype(int)

binary_pred = (
    y_pred != 0
).astype(int)


binary_metrics = calculate_metrics(

    binary_true,
    binary_pred,

    labels=[
        0,
        1,
    ],
)


print("\n" + "=" * 120)
print("POOLED H=10 ZERO-SHOT BINARY GOOD-vs-DEFECT PERFORMANCE")
print("=" * 120)


for key, value in binary_metrics.items():

    if key == "n_samples":

        print(
            f"{key:25s}: {value}"
        )

    else:

        print(
            f"{key:25s}: {value * 100:.2f}%"
        )


print("\nBinary classification report:")

print(
    classification_report(

        binary_true,
        binary_pred,

        labels=[
            0,
            1,
        ],

        target_names=[
            "Good",
            "Defect",
        ],

        zero_division=0,

        digits=4,
    )
)


# ==================================================================================================
# 22. BINARY CONFUSION MATRIX
# ==================================================================================================

binary_cm = confusion_matrix(

    binary_true,
    binary_pred,

    labels=[
        0,
        1,
    ],
)


print(
    "\nRaw binary confusion matrix:"
)

print(
    binary_cm
)


pd.DataFrame(

    binary_cm,

    index=[
        "Good",
        "Defect",
    ],

    columns=[
        "Good",
        "Defect",
    ],

).to_csv(

    OUTPUT_DIR
    / "pooled_binary_confusion_matrix_RAW.csv"
)


plot_confusion_matrix(

    binary_cm,

    [
        "Good",
        "Defect",
    ],

    "Independent H=10 zero-shot binary evaluation — Exp. 12–14",

    OUTPUT_DIR
    / "pooled_binary_confusion_matrix.png",
)


# ==================================================================================================
# 23. NORMALIZED BINARY CONFUSION MATRIX
# ==================================================================================================

binary_row_sums = binary_cm.sum(
    axis=1,
    keepdims=True,
)


binary_normalized = np.divide(

    binary_cm,

    binary_row_sums,

    out=np.zeros_like(
        binary_cm,
        dtype=float,
    ),

    where=(
        binary_row_sums != 0
    ),

) * 100


print(
    "\nNormalized binary confusion matrix (%):"
)

print(
    np.round(
        binary_normalized,
        2,
    )
)


pd.DataFrame(

    binary_normalized,

    index=[
        "Good",
        "Defect",
    ],

    columns=[
        "Good",
        "Defect",
    ],

).to_csv(

    OUTPUT_DIR
    / "pooled_binary_confusion_matrix_NORMALIZED.csv"
)


# ==================================================================================================
# 24. PER-EXPERIMENT PERFORMANCE
# ==================================================================================================

print("\n" + "=" * 120)
print("PER-EXPERIMENT H=10 ZERO-SHOT PERFORMANCE")
print("=" * 120)


summary_rows = []


for exp_id in EXPERIMENTS:

    temp = evaluation_df[
        evaluation_df[
            "exp_id"
        ]
        == exp_id
    ]


    true_exp = temp[
        "true_class_id"
    ].astype(int)

    pred_exp = temp[
        "pred_class_id"
    ].astype(int)


    multi = calculate_metrics(

        true_exp,
        pred_exp,

        labels=CLASS_IDS,
    )


    true_binary_exp = (
        true_exp != 0
    ).astype(int)

    pred_binary_exp = (
        pred_exp != 0
    ).astype(int)


    binary = calculate_metrics(

        true_binary_exp,
        pred_binary_exp,

        labels=[
            0,
            1,
        ],
    )


    print(
        f"\n{exp_id.upper()}"
    )

    print(
        f"Samples               : {len(temp)}"
    )

    print(
        f"Multiclass Accuracy   : {multi['accuracy'] * 100:.2f}%"
    )

    print(
        f"Multiclass Macro F1   : {multi['macro_f1'] * 100:.2f}%"
    )

    print(
        f"Binary Accuracy       : {binary['accuracy'] * 100:.2f}%"
    )

    print(
        f"Binary Macro F1       : {binary['macro_f1'] * 100:.2f}%"
    )


    summary_rows.append(
        {
            "experiment":
                exp_id,

            "n_samples":
                len(temp),

            "multiclass_accuracy":
                multi[
                    "accuracy"
                ] * 100,

            "multiclass_macro_precision":
                multi[
                    "macro_precision"
                ] * 100,

            "multiclass_macro_recall":
                multi[
                    "macro_recall"
                ] * 100,

            "multiclass_macro_f1":
                multi[
                    "macro_f1"
                ] * 100,

            "multiclass_weighted_f1":
                multi[
                    "weighted_f1"
                ] * 100,

            "binary_accuracy":
                binary[
                    "accuracy"
                ] * 100,

            "binary_macro_precision":
                binary[
                    "macro_precision"
                ] * 100,

            "binary_macro_recall":
                binary[
                    "macro_recall"
                ] * 100,

            "binary_macro_f1":
                binary[
                    "macro_f1"
                ] * 100,

            "binary_weighted_f1":
                binary[
                    "weighted_f1"
                ] * 100,
        }
    )


# ==================================================================================================
# 25. ADD POOLED RESULTS
# ==================================================================================================

summary_rows.append(
    {
        "experiment":
            "POOLED_EXP_12_13_14",

        "n_samples":
            len(evaluation_df),

        "multiclass_accuracy":
            multiclass_metrics[
                "accuracy"
            ] * 100,

        "multiclass_macro_precision":
            multiclass_metrics[
                "macro_precision"
            ] * 100,

        "multiclass_macro_recall":
            multiclass_metrics[
                "macro_recall"
            ] * 100,

        "multiclass_macro_f1":
            multiclass_metrics[
                "macro_f1"
            ] * 100,

        "multiclass_weighted_f1":
            multiclass_metrics[
                "weighted_f1"
            ] * 100,

        "binary_accuracy":
            binary_metrics[
                "accuracy"
            ] * 100,

        "binary_macro_precision":
            binary_metrics[
                "macro_precision"
            ] * 100,

        "binary_macro_recall":
            binary_metrics[
                "macro_recall"
            ] * 100,

        "binary_macro_f1":
            binary_metrics[
                "macro_f1"
            ] * 100,

        "binary_weighted_f1":
            binary_metrics[
                "weighted_f1"
            ] * 100,
    }
)


summary_df = pd.DataFrame(
    summary_rows
)


summary_df.to_csv(

    OUTPUT_DIR
    / "FINAL_H10_ZERO_SHOT_PERFORMANCE_SUMMARY.csv",

    index=False,
)


# ==================================================================================================
# 26. FINAL SUMMARY
# ==================================================================================================

print("\n" + "=" * 120)
print("FINAL MANUSCRIPT-READY ZERO-SHOT SUMMARY")
print("=" * 120)


print(
    summary_df
    .round(2)
    .to_string(
        index=False
    )
)


print("\nGround-truth construction:")
print(
    "  No recognized defect -> Good"
)

print(
    "  One or more recognized defects -> class with largest independent COCO annotation area"
)


print("\nEvaluation samples:")
print(
    len(evaluation_df)
)


print("\nPOOLED MULTICLASS:")
print(
    f"Accuracy        : {multiclass_metrics['accuracy'] * 100:.2f}%"
)

print(
    f"Macro Precision : {multiclass_metrics['macro_precision'] * 100:.2f}%"
)

print(
    f"Macro Recall    : {multiclass_metrics['macro_recall'] * 100:.2f}%"
)

print(
    f"Macro F1        : {multiclass_metrics['macro_f1'] * 100:.2f}%"
)

print(
    f"Weighted F1     : {multiclass_metrics['weighted_f1'] * 100:.2f}%"
)


print("\nPOOLED BINARY GOOD-vs-DEFECT:")
print(
    f"Accuracy        : {binary_metrics['accuracy'] * 100:.2f}%"
)

print(
    f"Macro Precision : {binary_metrics['macro_precision'] * 100:.2f}%"
)

print(
    f"Macro Recall    : {binary_metrics['macro_recall'] * 100:.2f}%"
)

print(
    f"Macro F1        : {binary_metrics['macro_f1'] * 100:.2f}%"
)

print(
    f"Weighted F1     : {binary_metrics['weighted_f1'] * 100:.2f}%"
)


print("\nResults saved to:")
print(
    OUTPUT_DIR
)


print("\n" + "=" * 120)
print("DONE")
print("=" * 120)